# Test Project Structure and Configuration

## Organizing Your Tests
---

In this lesson, you'll learn how to **properly organize test projects** and configure pytest for your needs.

You have tests, but where do they go? How do you share common setup? Let's find out.

1. [Directory Structure](#Directory-Structure),
    - [Standard Layout](#Standard-Layout),
    - [Alternative Layouts](#Alternative-Layouts),
    - [Naming Conventions](#Naming-Conventions),
2. [The conftest.py File](#The-conftest.py-File),
    - [What Is conftest.py](#What-Is-conftest.py),
    - [Shared Fixtures](#Shared-Fixtures),
    - [Fixture Scope](#Fixture-Scope),
    - [Multiple conftest Files](#Multiple-conftest-Files),
3. [Test Functions vs Test Classes](#Test-Functions-vs-Test-Classes),
    - [When to Use Functions](#When-to-Use-Functions),
    - [When to Use Classes](#When-to-Use-Classes),
    - [Mixing Both](#Mixing-Both),
4. [pytest Configuration](#pytest-Configuration),
    - [pyproject.toml](#pyproject.toml),
    - [pytest.ini](#pytest.ini),
    - [Common Options](#Common-Options),
5. [Running Specific Tests](#Running-Specific-Tests),
    - [By File and Directory](#By-File-and-Directory),
    - [By Name Pattern](#By-Name-Pattern),
    - [By Marker](#By-Marker),
6. [Practical Example: E-commerce Test Suite](#Practical-Example:-E-commerce-Test-Suite),
7. [🧠 EXERCISE 🧠, Organize a Test Suite](#🧠-EXERCISE-🧠,-Organize-a-Test-Suite).

<br>

## Directory Structure

---

A well-organized test structure makes your project maintainable and your tests easy to find.

<br>

### Standard Layout

---

The most common and recommended structure:

```
my_project/
├── src/
│   └── my_package/
│       ├── __init__.py
│       ├── models.py
│       ├── services.py
│       └── utils.py
├── tests/
│   ├── __init__.py           # Makes tests a package (optional but recommended)
│   ├── conftest.py           # Shared fixtures and configuration
│   ├── test_models.py        # Tests for models.py
│   ├── test_services.py      # Tests for services.py
│   └── test_utils.py         # Tests for utils.py
├── pyproject.toml            # Project and pytest configuration
└── .venv/
```

**Key points:**

- Tests are in a separate `tests/` directory at project root
- Test files mirror the source structure
- `conftest.py` contains shared test utilities
- Configuration in `pyproject.toml`

<br>

### Alternative Layouts

---

**Nested tests (for larger projects):**

```
my_project/
├── src/
│   └── my_package/
│       ├── auth/
│       │   ├── __init__.py
│       │   ├── login.py
│       │   └── permissions.py
│       └── orders/
│           ├── __init__.py
│           ├── models.py
│           └── processing.py
├── tests/
│   ├── conftest.py              # Root-level fixtures
│   ├── auth/
│   │   ├── conftest.py          # Auth-specific fixtures
│   │   ├── test_login.py
│   │   └── test_permissions.py
│   └── orders/
│       ├── conftest.py          # Order-specific fixtures
│       ├── test_models.py
│       └── test_processing.py
└── pyproject.toml
```

**Tests alongside source (less common):**

```
my_project/
├── src/
│   └── my_package/
│       ├── __init__.py
│       ├── models.py
│       ├── test_models.py    # Tests next to source
│       ├── services.py
│       └── test_services.py
└── pyproject.toml
```

This approach is less common but can be useful for small projects or when tests are tightly coupled to implementation.

<br>

### Naming Conventions

---

pytest auto-discovers tests based on these conventions:

| Element | Convention | Example |
|---------|------------|----------|
| **Test directories** | `tests/`, `test/` | `tests/` |
| **Test files** | `test_*.py`, `*_test.py` | `test_users.py` |
| **Test functions** | `test_*` | `test_create_user()` |
| **Test classes** | `Test*` (no `__init__`) | `TestUserCreation` |
| **Test methods** | `test_*` | `test_valid_email()` |

In [ ]:
# Good naming examples

# test_users.py - tests for users module
def test_create_user_with_valid_data():
    """Test user creation with valid input."""
    pass

def test_create_user_rejects_duplicate_email():
    """Test that duplicate emails are rejected."""
    pass

def test_user_password_is_hashed():
    """Test that passwords are properly hashed."""
    pass


class TestUserPermissions:
    """Tests for user permission system."""
    
    def test_admin_has_all_permissions(self):
        pass
    
    def test_regular_user_has_limited_permissions(self):
        pass

<br>

## The conftest.py File

---

### What Is conftest.py

---

`conftest.py` is a special file recognized by pytest. It contains:

- **Fixtures** - reusable test setup/teardown
- **Hooks** - customize pytest behavior
- **Plugins** - local pytest plugins

**Key features:**
- No need to import - fixtures are auto-discovered
- Can have multiple `conftest.py` files at different levels
- Fixtures are available to all tests in same directory and below

<br>

### Shared Fixtures

---

A **fixture** provides test data or test setup that can be reused across tests:

In [ ]:
# tests/conftest.py

import pytest


@pytest.fixture
def sample_user() -> dict:
    """Provide a sample user for testing."""
    return {
        "username": "testuser",
        "email": "test@example.com",
        "password": "securepassword123",
        "age": 25,
    }


@pytest.fixture
def sample_product() -> dict:
    """Provide a sample product for testing."""
    return {
        "id": "PROD-001",
        "name": "Test Product",
        "price": 29.99,
        "stock": 100,
    }

In [ ]:
# tests/test_users.py

def test_user_creation(sample_user):
    """Test using the sample_user fixture."""
    # sample_user is automatically injected by pytest
    assert sample_user["username"] == "testuser"
    assert sample_user["email"] == "test@example.com"


def test_user_email_validation(sample_user):
    """Another test using the same fixture."""
    assert "@" in sample_user["email"]

<br>

**Fixtures with setup and teardown:**

In [ ]:
# tests/conftest.py

import pytest
import tempfile
import os


@pytest.fixture
def temp_config_file():
    """Create a temporary config file, clean up after test."""
    # Setup: create temp file
    fd, path = tempfile.mkstemp(suffix=".json")
    with os.fdopen(fd, "w") as f:
        f.write('{"debug": true, "timeout": 30}')
    
    # Provide the path to the test
    yield path
    
    # Teardown: remove temp file
    if os.path.exists(path):
        os.remove(path)

In [ ]:
# tests/test_config.py

import json


def test_load_config(temp_config_file):
    """Test config loading from file."""
    with open(temp_config_file) as f:
        config = json.load(f)
    
    assert config["debug"] == True
    assert config["timeout"] == 30
    
    # After test: file is automatically cleaned up

<br>

### Fixture Scope

---

Fixtures can have different scopes controlling when they're created/destroyed:

| Scope | When Created | When Destroyed | Use Case |
|-------|--------------|----------------|----------|
| `function` (default) | Before each test | After each test | Test isolation |
| `class` | Before first test in class | After last test in class | Class-level setup |
| `module` | Before first test in file | After last test in file | Expensive setup |
| `session` | Once at start | Once at end | Database connection |

In [ ]:
# tests/conftest.py

import pytest


@pytest.fixture(scope="function")  # Default - runs for each test
def fresh_data():
    """New data for each test - ensures isolation."""
    return {"counter": 0}


@pytest.fixture(scope="module")
def expensive_resource():
    """Created once per test file - for expensive operations."""
    print("Creating expensive resource...")
    resource = {"data": "loaded from slow source"}
    yield resource
    print("Cleaning up expensive resource...")


@pytest.fixture(scope="session")
def database_connection():
    """Created once for entire test session."""
    print("Connecting to database...")
    connection = {"connected": True}  # Simulated connection
    yield connection
    print("Closing database connection...")

<br>

### Multiple conftest Files

---

You can have `conftest.py` at different directory levels:

```
tests/
├── conftest.py              # Available to ALL tests
│
├── unit/
│   ├── conftest.py          # Available only to unit tests
│   ├── test_models.py
│   └── test_utils.py
│
└── integration/
    ├── conftest.py          # Available only to integration tests
    ├── test_api.py
    └── test_database.py
```

In [ ]:
# tests/conftest.py - root level

import pytest

@pytest.fixture
def base_url() -> str:
    """API base URL - available to all tests."""
    return "http://localhost:8000"

In [ ]:
# tests/unit/conftest.py - unit tests only

import pytest

@pytest.fixture
def mock_database():
    """Mock database for unit tests - no real DB needed."""
    return {"users": [], "orders": []}

In [ ]:
# tests/integration/conftest.py - integration tests only

import pytest

@pytest.fixture(scope="module")
def test_database():
    """Real test database for integration tests."""
    # Setup test database
    db = create_test_database()
    yield db
    # Teardown
    db.drop_all()

<br>

## Test Functions vs Test Classes

---

### When to Use Functions

---

**Use standalone functions** for:
- Simple, independent tests
- Tests that don't share setup
- Quick one-off tests

In [ ]:
# tests/test_validators.py

def test_email_valid():
    assert validate_email("user@example.com") == True

def test_email_missing_at():
    assert validate_email("userexample.com") == False

def test_email_empty():
    assert validate_email("") == False

def test_username_valid():
    assert validate_username("john_doe") == True

def test_username_too_short():
    assert validate_username("ab") == False

<br>

### When to Use Classes

---

**Use test classes** for:
- Grouping related tests
- Tests that share setup/context
- Testing a specific component/feature

In [ ]:
# tests/test_order.py

import pytest
from decimal import Decimal


class TestOrderCreation:
    """Tests for order creation."""
    
    def test_create_empty_order(self):
        order = Order(customer_id=1)
        assert order.total == Decimal("0")
    
    def test_create_order_with_item(self):
        order = Order(customer_id=1)
        order.add_item(product_id=1, quantity=2, price=Decimal("10"))
        assert order.total == Decimal("20")


class TestOrderDiscounts:
    """Tests for order discounts."""
    
    @pytest.fixture
    def order_with_items(self):
        """Order with items for discount testing."""
        order = Order(customer_id=1)
        order.add_item(product_id=1, quantity=1, price=Decimal("100"))
        return order
    
    def test_apply_percentage_discount(self, order_with_items):
        order_with_items.apply_discount(percent=10)
        assert order_with_items.total == Decimal("90")
    
    def test_apply_fixed_discount(self, order_with_items):
        order_with_items.apply_discount(fixed=Decimal("15"))
        assert order_with_items.total == Decimal("85")


class TestOrderPayment:
    """Tests for order payment processing."""
    
    def test_mark_as_paid(self):
        order = Order(customer_id=1)
        order.mark_paid()
        assert order.status == "paid"
    
    def test_cannot_modify_paid_order(self):
        order = Order(customer_id=1)
        order.mark_paid()
        
        with pytest.raises(ValueError):
            order.add_item(product_id=1, quantity=1, price=Decimal("10"))

<br>

### Mixing Both

---

It's perfectly fine to mix functions and classes in the same file:

In [ ]:
# tests/test_users.py

# Simple standalone tests
def test_username_validation():
    assert is_valid_username("john_doe") == True


def test_email_validation():
    assert is_valid_email("john@example.com") == True


# Related tests grouped in class
class TestUserRegistration:
    """Tests for user registration flow."""
    
    def test_successful_registration(self):
        pass
    
    def test_duplicate_email_rejected(self):
        pass
    
    def test_weak_password_rejected(self):
        pass


class TestUserLogin:
    """Tests for user login."""
    
    def test_successful_login(self):
        pass
    
    def test_wrong_password(self):
        pass

<br>

## pytest Configuration

---

### pyproject.toml

---

The **recommended** way to configure pytest (modern Python projects):

In [ ]:
# pyproject.toml

# [tool.pytest.ini_options]
# testpaths = ["tests"]
# python_files = ["test_*.py", "*_test.py"]
# python_classes = ["Test*"]
# python_functions = ["test_*"]
# addopts = "-v --strict-markers"
# markers = [
#     "slow: marks tests as slow (deselect with '-m "not slow"')",
#     "integration: marks tests as integration tests",
# ]

**Full example `pyproject.toml`:**

```toml
[project]
name = "my-project"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = []

[project.optional-dependencies]
dev = [
    "pytest>=8.0.0",
    "pytest-cov>=4.0.0",
]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"
markers = [
    "slow: marks tests as slow",
    "integration: integration tests",
]

[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"
```

<br>

### pytest.ini

---

Alternative configuration file (legacy, but still supported):

```ini
# pytest.ini

[pytest]
testpaths = tests
python_files = test_*.py
python_classes = Test*
python_functions = test_*
addopts = -v --strict-markers
markers =
    slow: marks tests as slow
    integration: integration tests
```

<br>

### Common Options

---

| Option | Description | Example |
|--------|-------------|----------|
| `testpaths` | Directories to search | `["tests"]` |
| `addopts` | Default command line options | `"-v --tb=short"` |
| `markers` | Custom markers definition | `["slow: ..."]` |
| `filterwarnings` | Warning filters | `["ignore::DeprecationWarning"]` |
| `python_files` | Test file patterns | `["test_*.py"]` |
| `norecursedirs` | Skip these directories | `[".venv", "node_modules"]` |

**Commonly used `addopts`:**

```toml
[tool.pytest.ini_options]
addopts = """
    -v
    --tb=short
    --strict-markers
    -ra
"""
```

| Flag | Effect |
|------|--------|
| `-v` | Verbose output |
| `--tb=short` | Shorter tracebacks |
| `--strict-markers` | Error on unknown markers |
| `-ra` | Show summary of all non-passing tests |

<br>

## Running Specific Tests

---

### By File and Directory

---

**Linux/macOS:**

```bash
# Run all tests
pytest

# Run tests in specific directory
pytest tests/unit/

# Run tests in specific file
pytest tests/test_users.py

# Run specific test function
pytest tests/test_users.py::test_create_user

# Run specific test class
pytest tests/test_users.py::TestUserRegistration

# Run specific method in class
pytest tests/test_users.py::TestUserRegistration::test_valid_email
```

**Windows:**

```powershell
# Run all tests
pytest

# Run tests in specific directory
pytest tests\unit\

# Run tests in specific file
pytest tests\test_users.py

# Run specific test function
pytest tests\test_users.py::test_create_user

# Run specific test class
pytest tests\test_users.py::TestUserRegistration

# Run specific method in class
pytest tests\test_users.py::TestUserRegistration::test_valid_email
```

<br>

### By Name Pattern

---

Use `-k` to filter by test name pattern:

```bash
# Run tests containing 'user' in name
pytest -k "user"

# Run tests containing 'user' AND 'valid'
pytest -k "user and valid"

# Run tests containing 'user' but NOT 'invalid'
pytest -k "user and not invalid"

# Run tests containing 'email' OR 'password'
pytest -k "email or password"
```

<br>

### By Marker

---

Mark tests with decorators, then filter:

In [ ]:
# tests/test_integration.py

import pytest


@pytest.mark.slow
def test_large_data_processing():
    """This test takes a long time."""
    pass


@pytest.mark.integration
def test_database_connection():
    """This test requires real database."""
    pass


@pytest.mark.integration
@pytest.mark.slow
def test_full_workflow():
    """Slow integration test."""
    pass


def test_fast_unit():
    """Fast unit test - no markers."""
    pass

**Running with markers:**

```bash
# Run only slow tests
pytest -m slow

# Run only integration tests
pytest -m integration

# Skip slow tests
pytest -m "not slow"

# Run integration but not slow
pytest -m "integration and not slow"
```

**Built-in markers:**

| Marker | Purpose |
|--------|---------|
| `@pytest.mark.skip` | Skip test unconditionally |
| `@pytest.mark.skipif(condition)` | Skip if condition is true |
| `@pytest.mark.xfail` | Expected to fail |
| `@pytest.mark.parametrize` | Run with multiple inputs |

In [ ]:
import pytest
import sys


@pytest.mark.skip(reason="Not implemented yet")
def test_future_feature():
    pass


@pytest.mark.skipif(sys.platform == "win32", reason="Unix only")
def test_unix_specific():
    pass


@pytest.mark.xfail(reason="Known bug #123")
def test_known_issue():
    assert False  # This failure is expected

<br>

## Practical Example: E-commerce Test Suite

---

Let's see a complete test structure for an e-commerce project:

```
ecommerce/
├── src/
│   └── ecommerce/
│       ├── __init__.py
│       ├── models/
│       │   ├── __init__.py
│       │   ├── product.py
│       │   ├── user.py
│       │   └── order.py
│       ├── services/
│       │   ├── __init__.py
│       │   ├── cart.py
│       │   ├── checkout.py
│       │   └── payment.py
│       └── utils/
│           ├── __init__.py
│           └── validators.py
│
├── tests/
│   ├── conftest.py                    # Shared fixtures
│   │
│   ├── unit/
│   │   ├── conftest.py                # Unit test fixtures
│   │   ├── models/
│   │   │   ├── test_product.py
│   │   │   ├── test_user.py
│   │   │   └── test_order.py
│   │   ├── services/
│   │   │   ├── test_cart.py
│   │   │   └── test_checkout.py
│   │   └── utils/
│   │       └── test_validators.py
│   │
│   └── integration/
│       ├── conftest.py                # Integration fixtures (DB, etc.)
│       ├── test_checkout_flow.py
│       └── test_payment_processing.py
│
└── pyproject.toml
```

In [ ]:
# tests/conftest.py - Root level, available to ALL tests

import pytest
from decimal import Decimal


@pytest.fixture
def sample_product_data() -> dict:
    """Standard product data for testing."""
    return {
        "id": "PROD-001",
        "name": "Test Widget",
        "price": Decimal("29.99"),
        "stock": 100,
    }


@pytest.fixture
def sample_user_data() -> dict:
    """Standard user data for testing."""
    return {
        "id": "USER-001",
        "email": "test@example.com",
        "name": "Test User",
    }

In [ ]:
# tests/unit/conftest.py - Unit tests only

import pytest


@pytest.fixture
def mock_payment_gateway():
    """Mock payment gateway for unit tests."""
    class MockGateway:
        def charge(self, amount, card):
            return {"status": "success", "transaction_id": "TX-123"}
    
    return MockGateway()

In [ ]:
# tests/integration/conftest.py - Integration tests only

import pytest


@pytest.fixture(scope="session")
def test_database():
    """Create test database for integration tests."""
    db = setup_test_database()
    yield db
    teardown_test_database(db)


@pytest.fixture
def payment_gateway():
    """Real (test mode) payment gateway."""
    return PaymentGateway(test_mode=True)

In [ ]:
# tests/unit/services/test_cart.py

import pytest
from decimal import Decimal
from src.ecommerce.services.cart import Cart


class TestCart:
    """Tests for shopping cart."""
    
    def test_empty_cart(self):
        cart = Cart()
        assert cart.total == Decimal("0")
        assert len(cart.items) == 0
    
    def test_add_item(self, sample_product_data):
        cart = Cart()
        cart.add_item(
            product_id=sample_product_data["id"],
            price=sample_product_data["price"],
            quantity=2
        )
        
        assert len(cart.items) == 1
        assert cart.total == Decimal("59.98")
    
    def test_remove_item(self, sample_product_data):
        cart = Cart()
        cart.add_item(
            product_id=sample_product_data["id"],
            price=sample_product_data["price"],
            quantity=1
        )
        cart.remove_item(sample_product_data["id"])
        
        assert len(cart.items) == 0

**Configuration (`pyproject.toml`):**

```toml
[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --strict-markers -ra"
markers = [
    "slow: marks tests as slow (deselect with '-m \"not slow\"')",
    "integration: marks tests as integration tests",
]
filterwarnings = [
    "ignore::DeprecationWarning",
]
```

**Running the tests:**

```bash
# All tests
pytest

# Only unit tests
pytest tests/unit/

# Only integration tests
pytest tests/integration/

# Skip slow tests
pytest -m "not slow"

# Only cart-related tests
pytest -k "cart"
```

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Directory structure** | `tests/` directory, mirror source structure |
| **conftest.py** | Shared fixtures, auto-discovered, multi-level |
| **Fixture scope** | `function`, `class`, `module`, `session` |
| **Functions vs classes** | Functions for simple tests, classes for grouping |
| **Configuration** | `pyproject.toml` (recommended) or `pytest.ini` |
| **Running tests** | `-k` for name pattern, `-m` for markers |

<br>

**Key principles:**

1. **Mirror your source structure** in tests
2. **Use conftest.py** for shared fixtures
3. **Choose appropriate fixture scope** for performance
4. **Group related tests** in classes
5. **Use markers** to categorize tests
6. **Configure pytest** in `pyproject.toml`

<br>

### 🧠 EXERCISE 🧠, Organize a Test Suite

---

You're given a flat test file. Reorganize it into a proper structure.

<br>

**Original messy test file** (`test_everything.py`):

In [ ]:
# test_everything.py - BEFORE (messy)

from decimal import Decimal

def test_product_creation():
    product = Product(name="Widget", price=Decimal("10"))
    assert product.name == "Widget"

def test_user_email_validation():
    assert validate_email("test@example.com") == True

def test_order_total():
    order = Order()
    order.add_item(Product("A", Decimal("10")), 2)
    assert order.total == Decimal("20")

def test_product_price_validation():
    # Can't have negative price
    pass

def test_user_password_hashing():
    pass

def test_order_discount():
    pass

def test_user_registration():
    pass

def test_product_stock_update():
    pass

def test_order_checkout_flow():
    # Integration test - needs real DB
    pass

<br>

**Your task:**

1. Create a proper directory structure with:
   - `tests/` directory
   - `tests/unit/` for unit tests
   - `tests/integration/` for integration tests
   - Appropriate `conftest.py` files

2. Split tests into separate files:
   - `test_product.py` - product-related tests
   - `test_user.py` - user-related tests
   - `test_order.py` - order-related tests

3. Create shared fixtures in `conftest.py`

4. Add a `pyproject.toml` with pytest configuration

<details>
    <summary>▶️ Solution</summary>

**Directory structure:**

```
my_project/
├── src/
│   └── shop/
│       ├── __init__.py
│       ├── models.py
│       └── validators.py
├── tests/
│   ├── conftest.py
│   ├── unit/
│   │   ├── conftest.py
│   │   ├── test_product.py
│   │   ├── test_user.py
│   │   └── test_order.py
│   └── integration/
│       ├── conftest.py
│       └── test_checkout.py
└── pyproject.toml
```

---

**tests/conftest.py:**

```python
import pytest
from decimal import Decimal


@pytest.fixture
def sample_product():
    """Sample product for testing."""
    from src.shop.models import Product
    return Product(name="Test Widget", price=Decimal("29.99"))


@pytest.fixture
def sample_user():
    """Sample user for testing."""
    return {
        "email": "test@example.com",
        "password": "securepass123",
        "name": "Test User",
    }
```

---

**tests/unit/test_product.py:**

```python
import pytest
from decimal import Decimal
from src.shop.models import Product


class TestProductCreation:
    """Tests for product creation."""
    
    def test_create_product(self):
        product = Product(name="Widget", price=Decimal("10"))
        assert product.name == "Widget"
        assert product.price == Decimal("10")
    
    def test_negative_price_rejected(self):
        with pytest.raises(ValueError):
            Product(name="Widget", price=Decimal("-10"))


class TestProductStock:
    """Tests for product stock management."""
    
    def test_update_stock(self, sample_product):
        sample_product.update_stock(50)
        assert sample_product.stock == 50
```

---

**tests/unit/test_user.py:**

```python
import pytest
from src.shop.validators import validate_email
from src.shop.models import User


class TestUserValidation:
    """Tests for user validation."""
    
    def test_valid_email(self):
        assert validate_email("test@example.com") == True
    
    def test_invalid_email(self):
        assert validate_email("invalid") == False


class TestUserRegistration:
    """Tests for user registration."""
    
    def test_password_is_hashed(self, sample_user):
        user = User(**sample_user)
        assert user.password != sample_user["password"]
```

---

**tests/unit/test_order.py:**

```python
import pytest
from decimal import Decimal
from src.shop.models import Order


class TestOrderTotal:
    """Tests for order total calculation."""
    
    def test_single_item_total(self, sample_product):
        order = Order()
        order.add_item(sample_product, quantity=2)
        assert order.total == Decimal("59.98")


class TestOrderDiscount:
    """Tests for order discounts."""
    
    def test_percentage_discount(self, sample_product):
        order = Order()
        order.add_item(sample_product, quantity=1)
        order.apply_discount(percent=10)
        
        assert order.total == pytest.approx(Decimal("26.99"), rel=0.01)
```

---

**tests/integration/conftest.py:**

```python
import pytest


@pytest.fixture(scope="module")
def test_database():
    """Setup test database."""
    db = create_test_db()
    yield db
    db.drop_all()
```

---

**tests/integration/test_checkout.py:**

```python
import pytest


@pytest.mark.integration
class TestCheckoutFlow:
    """Integration tests for checkout."""
    
    def test_complete_checkout(self, test_database, sample_product, sample_user):
        """Test complete checkout flow."""
        # Create order, process payment, update stock...
        pass
```

---

**pyproject.toml:**

```toml
[project]
name = "shop"
version = "0.1.0"

[project.optional-dependencies]
dev = ["pytest>=8.0.0"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --strict-markers"
markers = [
    "integration: integration tests (require database)",
]
```

</details>

---